# GNN-MARL Network Slicing — Training Notebook

Jalankan cell satu per satu dari atas ke bawah.

| Hardware | PPO (1M steps) | DQN (200K steps) |
|---|---|---|
| Colab T4   | ~45 menit/algo | ~2 jam/algo  |
| Colab A100 | ~15 menit/algo | ~45 menit/algo |

> **Sebelum mulai:** Runtime → Change runtime type → **GPU** (T4 gratis, A100 Colab Pro)  
> Hasil disimpan ke Google Drive — aman kalau session disconnect.  
> Kalau reconnect: jalankan ulang semua cell dari atas; cell yang sudah selesai **auto-skip**.

## Cell 1 — Mount Drive + Clone Repo

In [1]:
import os, sys, subprocess, time, csv, glob

# ── GANTI INI ────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/kruwpuck/gnn-marl-network-slicing.git"
DRIVE_DIR = "/content/drive/MyDrive/Semester 7/BU SOFI/gnn-marl-network-slicing"
# ─────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(f"{DRIVE_DIR}/.git"):
    subprocess.run(["git", "clone", REPO_URL, DRIVE_DIR], check=True)
else:
    subprocess.run(["git", "-C", DRIVE_DIR, "pull"], check=True)

os.chdir(DRIVE_DIR)
print("CWD:", os.getcwd())

Mounted at /content/drive
CWD: /content/drive/Othercomputers/My Laptop/Semester 7/BU SOFI/gnn-marl-network-slicing


## Cell 2 — Install Dependencies

In [2]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.4.0", "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch-geometric==2.8.0",
    "numpy==1.26.4", "scipy==1.13.0",
    "gymnasium==1.1.0", "pettingzoo==1.25.0", "pyyaml==6.0.2"
], check=True)

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    raise RuntimeError("GPU not found! Runtime → Change runtime type → GPU")

PyTorch : 2.4.0+cu124
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB


## Cell 3 — Smoke Test (70 tests)

In [3]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-500:])
    raise RuntimeError("Tests gagal — cek output")

......................................................................   [100%]
70 passed in 36.30s



## Cell 4 — Helper Functions

In [4]:
def _csv_done(algo, backbone=None, seed=42, min_ep=10):
    name = f"{algo}_{backbone}_seed{seed}.csv" if backbone else f"{algo}_seed{seed}.csv"
    path = f"results/logs/{name}"
    if not os.path.exists(path):
        return False
    with open(path) as f:
        return sum(1 for _ in f) - 1 >= min_ep


def train(algo, backbone=None, steps=1_000_000, seed=42):
    label = f"{algo}/{backbone}" if backbone else algo
    if _csv_done(algo, backbone, seed):
        print(f"[SKIP]  {label} — sudah ada CSV")
        return
    os.makedirs("results/logs", exist_ok=True)
    if backbone:
        cmd = [sys.executable, "training/train_proposed.py",
               "--algo", algo, "--backbone", backbone,
               "--steps", str(steps), "--seed", str(seed)]
    else:
        cmd = [sys.executable, "training/train_baselines.py",
               "--algo", algo, "--steps", str(steps), "--seed", str(seed)]
    print(f"\n{'='*55}\n  {label}  ({steps:,} steps)\n{'='*55}")
    t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f"  Selesai dalam {(time.time()-t0)/60:.1f} menit")


def check_progress():
    log_dir = "results/logs"
    if not os.path.exists(log_dir):
        print("Belum ada logs."); return
    print(f"{'File':<45} {'Episodes':>10} {'Last Step':>12} {'Last Reward':>12}")
    print("-" * 85)
    for f in sorted(glob.glob(f"{log_dir}/*.csv")):
        with open(f) as fp:
            rows = list(csv.reader(fp))
        n = len(rows) - 1
        if n > 0:
            last = rows[-1]
            rew = last[2] if len(last) > 2 else "-"
            print(f"{os.path.basename(f):<45} {n:>10} {last[0]:>12} {rew:>12}")
        else:
            print(f"{os.path.basename(f):<45} {'(empty)':>10}")


print("Helper functions loaded.")

Helper functions loaded.


## Cell 5 — PPO Training ⚡

Paling cepat. Estimasi T4: **~3 jam total** (4 algo × 45 menit).

In [5]:
train("gnn-mappo", backbone="gat",  steps=1_000_000)  # GNN-MAPPO / GAT

[SKIP]  gnn-mappo/gat — sudah ada CSV


In [6]:
train("gnn-mappo", backbone="sage", steps=1_000_000)  # GNN-MAPPO / SAGE

[SKIP]  gnn-mappo/sage — sudah ada CSV


In [7]:
train("ippo",        steps=1_000_000)  # IPPO baseline (MLP, independent)

[SKIP]  ippo — sudah ada CSV


In [8]:
train("central-ppo", steps=1_000_000)  # Central-PPO baseline (MLP, centralized)

[SKIP]  central-ppo — sudah ada CSV


## Cell 6 — DQN Training 🐢

CPU-bottlenecked (env simulation). 200K steps cukup untuk convergence comparison.

Estimasi T4: **~2 jam per algo**. Kalau khawatir timeout: jalankan 1 cell per sesi Colab.

In [9]:
train("gnn-madqn", backbone="gat",  steps=200_000)  # GNN-MADQN / GAT

[SKIP]  gnn-madqn/gat — sudah ada CSV


In [10]:
train("gnn-madqn", backbone="sage", steps=200_000)  # GNN-MADQN / SAGE

[SKIP]  gnn-madqn/sage — sudah ada CSV


In [11]:
train("idqn",        steps=200_000)  # IDQN baseline (MLP, independent)

[SKIP]  idqn — sudah ada CSV


In [12]:
train("central-dqn", steps=200_000)  # Central-DQN baseline (MLP, centralized)

[SKIP]  central-dqn — sudah ada CSV


## Cell 7 — Cek Progress (kapan saja)

In [13]:
check_progress()

File                                            Episodes    Last Step  Last Reward
-------------------------------------------------------------------------------------
central-dqn_seed42.csv                                59        11799     197.3544
central-ppo_seed42.csv                              1459       747007     -247.752
gnn-madqn_gat_seed42.csv                              14         2799      65.4163
gnn-madqn_sage_seed42.csv                             27         5399    -273.3075
gnn-mappo_gat_seed42.csv                             282       144383       81.433
gnn-mappo_sage_seed42.csv                            514       263167      165.143
idqn_seed42.csv                                       48         9599      -2.5456
ippo_seed42.csv                                     1507       771583    -410.0934


## Cell 8 — Evaluasi

Jalankan setelah semua training selesai. Butuh file `.pt` dari tiap algo.

In [14]:
# Phase 2a: Convergence curves → results/figures/
subprocess.run([
    sys.executable, "evaluation/convergence_eval.py",
    "--log-dir", "results/logs"
], check=True)

CompletedProcess(args=['/usr/bin/python3', 'evaluation/convergence_eval.py', '--log-dir', 'results/logs'], returncode=0)

In [15]:
# Phase 2b: Network performance (throughput, latency, Jain's fairness)
subprocess.run([sys.executable, "evaluation/network_perf_eval.py"], check=True)

CompletedProcess(args=['/usr/bin/python3', 'evaluation/network_perf_eval.py'], returncode=0)

In [16]:
# Phase 2c: Zero-shot generalization 5 → 10 → 20 gNB
subprocess.run([sys.executable, "evaluation/zero_shot_eval.py"], check=True)

CompletedProcess(args=['/usr/bin/python3', 'evaluation/zero_shot_eval.py'], returncode=0)

In [ ]:
# Phase 3: Ablation (backbone / reward weights / depth / burstiness)
for ablation in ["backbone", "reward", "depth", "burstiness"]:
    print(f"\n=== Ablation: {ablation} ===")
    subprocess.run([
        sys.executable, "ablation/backbone_ablation.py",
        "--ablation", ablation, "--steps", "200000"
    ], check=True)

print("\nSemua evaluasi selesai! Hasil di folder results/")


=== Ablation: backbone ===
